---
# **RUN on video**
---

In [1]:
import pyVHR as vhr
import numpy as np
from pyVHR.analysis.pipeline import Pipeline
from pyVHR.plot.visualize import *
import os
import plotly.express as px
from pyVHR.utils.errors import getErrors, printErrors, displayErrors

2024-07-08 17:52:04.176830: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0


This notebook shows the complete **pipeline** that takes a **video** as input and computes an **estimation** of **BPMs** using a specified **rPPG** **method**.

Below are the possible choices for the method.
* **Methods**: `cpu_CHROM`, `cupy_CHROM`, `torch_CHROM`, `cpu_LGI`, `cpu_POS`, `cupy_POS`, `cpu_PBV`, `cpu_PCA`, `cpu_GREEN`, `cpu_OMIT`, `cpu_ICA`, `cpu_SSR`

Below are the possible choices for the datasets.
* **Datasets**: `PURE`, `UBFC1`, `UBFC2`, `ECG_Fitness_01-1`, `ECG_Fitness_01-2`... `ECG_Fitness_06-1`, `ECG_Fitness_06-2`

#PURE dataset

In [2]:
# -- SET DATASET

dataset_name = 'PURE'                      # the name of the python class handling it 
video_DIR = '/var/datasets/VHR1/PURE/'  # dir containing videos
BVP_DIR = '/var/datasets/VHR1/PURE/'    # dir containing BVPs GT

dataset = vhr.datasets.datasetFactory(dataset_name, videodataDIR=video_DIR, BVPdataDIR=BVP_DIR)
allvideo = dataset.videoFilenames

# print the list of video names with the progressive index (idx)
for v in range(len(allvideo)):
  print(v, allvideo[v])

/var/datasets/VHR1/PURE/


In [42]:
# -- PARAMETER SETTING and GT

wsize = 4         # seconds of video processed (with overlapping) for each estimate 
video_idx = 1    # index of the video to be processed

# fname = dataset.getSigFilename(video_idx)
# sigGT = dataset.readSigfile(fname)
# test_bvp = sigGT.data
# bpmGT, timesGT = sigGT.getBPM(wsize)  # GT
# videoFileName = dataset.getVideoFilename(video_idx)
videoFileName = '/home/luying/pyVHR/pyVHR/videos/face2.mp4'
print('Video to process: ', videoFileName)
fps = vhr.extraction.get_fps(videoFileName)
print('Video frame rate: ',fps)

Video to process:  /home/luying/pyVHR/pyVHR/videos/face2.mp4
Video frame rate:  30.0


In [43]:
#### TEST holistic approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'median'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

configFilename='/home/luying/pyVHR/pyVHR/analysis/median.cfg'
res = pipe.run_on_dataset(configFilename, verb=True)
bpmGT = res.getData('bpmGT')
timesGT = res['timeGT']
# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

# CUDA devices:  2
# device number  0 :  NVIDIA GeForce GTX 1080 Ti
# device number  1 :  NVIDIA GeForce GTX 1080 Ti

Processing Video /home/luying/pyVHR/pyVHR/videos/face2.mp4

Roi processing...
 - Extraction approach: patches
 - Number of windows: 7
 - Win size: (#ROI, #landmarks, #frames) =  (100, 3, 120)

Pre filtering...
 - Pre-filter applied: BPfilter

BVP extraction...
 - Extraction method: cpu_CHROM
 - Post-filter applied: BPfilter

BPM estimation...
 - roi appproach: patches
 - BPM estimation with: median

...done!

** Run the test with the following config:
      dataset: LGI_PPGI
      methods: ['CUPY_CHROM_FILTER', 'TORCH_CHROM', 'CPU_LGI_FILTER', 'CPU_LGI', 'CUPY_POS_FILTER', 'CUPY_POS', 'CPU_PBV_FILTER', 'CPU_PBV', 'CPU_GREEN_FILTER', 'CPU_GREEN', 'CPU_PCA_SECOND_FILTER', 'CPU_PCA_SECOND', 'CPU_PCA_ALL_FILTER', 'CPU_PCA_ALL', 'CPU_ICA_SECOND_FILTER', 'CPU_ICA_SECOND', 'CPU_ICA_ALL_FILTER', 'CPU_ICA_ALL']
# CUDA devices:  2
# device number  0 :  NVIDIA GeForce GTX 1080 Ti


/home/luying/anaconda3/envs/pyvhr/lib/python3.9/site-packages/pyVHR/BVP/methods.py:39: RuntimeWarning: invalid value encountered in true_divide
  alpha = (sX/sY).reshape(-1, 1)


AttributeError: 'TestResult' object has no attribute 'getData'

In [39]:
#### TEST median approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'median'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)
res= pipe.run_on_dataset('/home/luying/pyVHR/results/cfg/ECG_Fitness_01-1_holistic.cfg', verb=True)
bmpGT = res['bpmGT']
timesGT = res['timesGT']

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

# CUDA devices:  2
# device number  0 :  NVIDIA GeForce GTX 1080 Ti
# device number  1 :  NVIDIA GeForce GTX 1080 Ti

Processing Video /home/luying/pyVHR/pyVHR/videos/face2.mp4

Roi processing...
 - Extraction approach: patches
 - Number of windows: 3
 - Win size: (#ROI, #landmarks, #frames) =  (100, 3, 240)

Pre filtering...
 - Pre-filter applied: BPfilter

BVP extraction...
 - Extraction method: cpu_CHROM
 - Post-filter applied: BPfilter

BPM estimation...
 - roi appproach: patches
 - BPM estimation with: median

...done!

** Run the test with the following config:
      dataset: ECG_FITNESS
      methods: ['CHROM', 'POS', 'LGI', 'ICA', 'PCA']
# CUDA devices:  2
# device number  0 :  NVIDIA GeForce GTX 1080 Ti
# device number  1 :  NVIDIA GeForce GTX 1080 Ti
 -  cuda device: True
 -  skin extractor: convexhull
 -  ROI approach: holistic


/home/luying/anaconda3/envs/pyvhr/lib/python3.9/site-packages/pyVHR/BVP/methods.py:39: RuntimeWarning: invalid value encountered in true_divide
  alpha = (sX/sY).reshape(-1, 1)


TypeError: 'TestResult' object is not subscriptable

In [34]:
#### TEST clustering approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'clustering'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

# CUDA devices:  2
# device number  0 :  NVIDIA GeForce GTX 1080 Ti
# device number  1 :  NVIDIA GeForce GTX 1080 Ti

Processing Video /home/luying/pyVHR/pyVHR/videos/face2.mp4

Roi processing...
 - Extraction approach: patches
 - Number of windows: 3
 - Win size: (#ROI, #landmarks, #frames) =  (100, 3, 240)

Pre filtering...
 - Pre-filter applied: BPfilter

BVP extraction...
 - Extraction method: cpu_CHROM
 - Post-filter applied: BPfilter

BPM estimation...
 - roi appproach: patches
 - BPM estimation with: clustering

...done!



NameError: name 'bpmGT' is not defined

#PURE dataset

In [15]:
# -- SET DATASET

dataset_name = 'PURE'                      # the name of the python class handling it 
video_DIR = '/var/datasets/VHR1/PURE/'  # dir containing videos
BVP_DIR = '/var/datasets/VHR1/PURE/'    # dir containing BVPs GT

dataset = vhr.datasets.datasetFactory(dataset_name, videodataDIR=video_DIR, BVPdataDIR=BVP_DIR)
allvideo = dataset.videoFilenames

# print the list of video names with the progressive index (idx)
for v in range(len(allvideo)):
  print(v, allvideo[v])

/var/datasets/VHR1/PURE/


In [16]:
# -- PARAMETER SETTING and GT

wsize = 8        # seconds of video processed (with overlapping) for each estimate 
video_idx = 11    # index of the video to be processed
fname = dataset.getSigFilename(video_idx)
sigGT = dataset.readSigfile(fname)
test_bvp = sigGT.data
bpmGT, timesGT = sigGT.getBPM(wsize)  # GT
videoFileName = dataset.getVideoFilename(video_idx)
print('Video to process: ', videoFileName)
fps = vhr.extraction.get_fps(videoFileName)
print('Video frame rate: ',fps)

IndexError: list index out of range

In [ ]:
#### TEST holistic approach

# params
roi_approach = 'holistic'   # 'holistic' or 'patches'
bpm_est = 'median'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

In [ ]:
#### TEST median approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'median'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

In [ ]:
#### TEST clustering approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'clustering'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

This section shows the complete pipeline that from a video calculates a time of BPM using a specified rPPG method.
* methods: `cpu_CHROM`, `cupy_CHROM`, `torch_CHROM`, `cpu_LGI`, `cpu_POS`, `cupy_POS`, `cpu_PBV`, `cpu_PCA`, `cpu_GREEN`, `cpu_OMIT`, `cpu_ICA`, `cpu_SSR`

Below are the possible choices for the datasets.
* datasets: `PURE`, `UBFC1`, `UBFC2`, `ECG_Fitness_01-1`, `ECG_Fitness_01-2`... `ECG_Fitness_06-1`, `ECG_Fitness_06-2`

#LGI-PPGI dataset

In [ ]:
# -- SET DATASET

dataset_name = 'LGI-PPGI'                   # the name of the python class handling it 
video_DIR = '/var/datasets/VHR1/LGI-PPGI/'  # dir containing videos
BVP_DIR = '/var/datasets/VHR1/LGI-PPGI/'    # dir containing BVPs GT

dataset = vhr.datasets.datasetFactory(dataset_name, videodataDIR=video_DIR, BVPdataDIR=BVP_DIR)
allvideo = dataset.videoFilenames

# print the list of video names with the progressive index (idx)
for v in range(len(allvideo)):
  print(v, allvideo[v])

In [ ]:
# -- PARAMETER SETTING and GT

wsize = 8        # seconds of video processed (with overlapping) for each estimate 
video_idx = 1    # index of the video to be processed
fname = dataset.getSigFilename(video_idx)
sigGT = dataset.readSigfile(fname)
test_bvp = sigGT.data
bpmGT, timesGT = sigGT.getBPM(wsize)  # GT
videoFileName = dataset.getVideoFilename(video_idx)
print('Video to process: ', videoFileName)
fps = vhr.extraction.get_fps(videoFileName)
print('Video frame rate: ',fps)

In [ ]:
#### TEST holistic approach

# params
roi_approach = 'holistic'   # 'holistic' or 'patches'
bpm_est = 'median'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

In [ ]:
#### TEST median approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'median'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

In [ ]:
#### TEST clustering approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'clustering'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

#UBFC1 dataset

In [ ]:
# -- SET DATASET

dataset_name = 'UBFC1'                   # the name of the python class handling it 
video_DIR = '/var/datasets/VHR1/UBFC1/'  # dir containing videos
BVP_DIR = '/var/datasets/VHR1/UBFC1/'    # dir containing BVPs GT

dataset = vhr.datasets.datasetFactory(dataset_name, videodataDIR=video_DIR, BVPdataDIR=BVP_DIR)
allvideo = dataset.videoFilenames

# print the list of video names with the progressive index (idx)
for v in range(len(allvideo)):
  print(v, allvideo[v])

In [ ]:
# -- PARAMETER SETTING and GT

wsize = 8        # seconds of video processed (with overlapping) for each estimate 
video_idx = 0    # index of the video to be processed
fname = dataset.getSigFilename(video_idx)
sigGT = dataset.readSigfile(fname)
test_bvp = sigGT.data
bpmGT, timesGT = sigGT.getBPM(wsize)  # GT
videoFileName = dataset.getVideoFilename(video_idx)
print('Video to process: ', videoFileName)
fps = vhr.extraction.get_fps(videoFileName)
print('Video frame rate: ',fps)

In [ ]:
#### TEST holistic approach

# params
roi_approach = 'holistic'   # 'holistic' or 'patches'
bpm_est = 'median'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

In [ ]:
#### TEST median approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'median'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

In [ ]:
#### TEST clustering approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'clustering'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True
)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

#ECG Fitness dataset

In [ ]:
# -- SET DATASET

dataset_name = 'ECG_Fitness'                   # the name of the python class handling it 
video_DIR = '/var/datasets/VHR1/ECG_Fitness_02-1/'  # dir containing videos
BVP_DIR = '/var/datasets/VHR1/ECG_Fitness_02-1/'    # dir containing BVPs GT

dataset = vhr.datasets.datasetFactory(dataset_name, videodataDIR=video_DIR, BVPdataDIR=BVP_DIR)
allvideo = dataset.videoFilenames

# print the list of video names with the progressive index (idx)
for v in range(len(allvideo)):
  print(v, allvideo[v])

In [ ]:
# -- PARAMETER SETTING and GT

wsize = 8        # seconds of video processed (with overlapping) for each estimate 
video_idx = 9    # index of the video to be processed
fname = dataset.getSigFilename(video_idx)
sigGT = dataset.readSigfile(fname)
test_bvp = sigGT.data
bpmGT, timesGT = sigGT.getBPM(wsize)  # GT
videoFileName = dataset.getVideoFilename(video_idx)
print('Video to process: ', videoFileName)
fps = vhr.extraction.get_fps(videoFileName)
print('Video frame rate: ',fps)

In [ ]:
#### TEST holistic approach

# params
roi_approach = 'holistic'   # 'holistic' or 'patches'
bpm_est = 'median'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=0, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

In [ ]:
#### TEST median approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'median'         # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        patch_size=40, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

In [ ]:
#### TEST clustering approach

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'clustering'     # BPM final estimate, if patches choose 'medians' or 'clustering'
method = 'cpu_CHROM'       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        method=method,
                                        estimate=bpm_est,
                                        movement_thrs=[20, 20, 10],
                                        patch_size=80, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True)

# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)

In [ ]:

# params
roi_approach = 'patches'   # 'holistic' or 'patches'
bpm_est = 'clustering'         # BPM final estimate, if patches choose 'medians' or 'clustering'
methods=['cpu_CHROM', 'cpu_LGI', 'cpu_ICA', 'cpu_PCA']       # one of the methods implemented in pyVHR
pipe = Pipeline()          # object to execute the pipeline

# run
bvps, timesES, bpmES = pipe.run_on_video_multimethods(videoFileName,
                                        winsize=wsize, 
                                        roi_method='convexhull',
                                        roi_approach=roi_approach,
                                        methods=methods,
                                        estimate=bpm_est,
                                        movement_thrs=[20, 20, 10],
                                        patch_size=80, 
                                        RGB_LOW_HIGH_TH=(5,230),
                                        Skin_LOW_HIGH_TH=(5,230),
                                        pre_filt=True,
                                        post_filt=True,
                                        cuda=True, 
                                        verb=True)


# ERRORS
RMSE, MAE, MAX, PCC, CCC, SNR = getErrors(bvps, fps, bpmES, bpmGT, timesES, timesGT)
printErrors(RMSE, MAE, MAX, PCC, CCC, SNR)
displayErrors(bpmES, bpmGT, timesES, timesGT)